# Gen AI Project: 2
### NxtWave Academy | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Generate your `submission.csv`

## Step 1: Install Required Libraries
Run this cell first if you are running in Kaggle, Colab, or a new environment.

In [1]:
# 1. INSTALL DEPENDENCIES
%pip install -q langchain langchain-community langchain-core langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf pandas langsmith
%pip install -q git+https://github.com/langchain-ai/langsmith-skills.git

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: git+https://github.com/langchain-ai/langsmith-skills.git does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


## Step 2: Import Dependencies

In [2]:
# 2. IMPORTS & DEPENDENCIES
import os
import csv
import time
import glob
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

try:
    from langsmith import traceable
except Exception:
    def traceable(*args, **kwargs):
        def decorator(func):
            return func
        if args and callable(args[0]):
            return args[0]
        return decorator

print("Dependencies imported successfully.")

Dependencies imported successfully.


## Step 3: Configure Groq API Key & Model
Set your Groq API key below or through environment variables / Kaggle secrets.

In [ ]:
# 3. CONFIGURATION & API KEY SETUP
# Use a Groq model that is available on the current API key.
LLM_MODEL = "openai/gpt-oss-20b"

# Read credentials from environment variables or Kaggle Secrets.
GROQ_API_KEY = (
    os.environ.get("GROQ_API_KEY")
    or os.environ.get("GROQ_KEY")
    or os.environ.get("groq_api_key")
)

if not GROQ_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
    except Exception:
        pass

# Local notebook fallback: prompt without writing the key to the file.
if not GROQ_API_KEY:
    try:
        from getpass import getpass
        GROQ_API_KEY = getpass("Enter your Groq API key (input hidden): ").strip()
    except Exception:
        pass

if not GROQ_API_KEY:
    raise ValueError(
        "Groq API key not found. Set GROQ_API_KEY or GROQ_KEY in your environment, "
        "add GROQ_API_KEY to Kaggle Secrets, or enter it when prompted."
    )

# LangSmith tracing setup. Keep this optional for local runs.
LANGSMITH_API_KEY = (
    os.environ.get("LANGSMITH_API_KEY")
    or os.environ.get("LANGCHAIN_API_KEY")
)

if LANGSMITH_API_KEY:
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = "zyro-rag-challenge"

print("=" * 50)
print(f"Provider: Groq | Model: {LLM_MODEL}")
print("Groq API key: Configured")
print(f"LangSmith tracing: {'Enabled' if LANGSMITH_API_KEY else 'Disabled'}")
print("LangSmith project: zyro-rag-challenge")
print("=" * 50)

ValueError: Groq API key not found. Set it as GROQ_API_KEY or GROQ_KEY in your environment or add it to Kaggle Secrets.

## Step 4: Locate & Load HR Policy Documents
Finds the HR policy PDFs in Kaggle or local directory, with automatic fallback context for testing.

In [ ]:
# 4. LOCATE & LOAD HR POLICY DOCUMENTS
candidate_paths = [
    "/kaggle/input/zyro-dynamics-hr-corpus/",
    "/kaggle/input/project-2-intelligent-rag/zyro-dynamics-hr-corpus/",
    "/kaggle/input/project-2-intelligent-rag/",
    "./hr_docs",
    "hr_docs",
    os.path.join(os.getcwd(), "hr_docs")
]

corpus_path = None
for cp in candidate_paths:
    if os.path.isdir(cp) and len(glob.glob(os.path.join(cp, "*.pdf"))) > 0:
        corpus_path = cp
        break

if not corpus_path:
    found_pdfs = glob.glob("**/*.pdf", recursive=True)
    if found_pdfs:
        corpus_path = os.path.dirname(found_pdfs[0])

documents = []
if corpus_path and os.path.isdir(corpus_path):
    print(f"Loading HR Policy documents from: {corpus_path}")
    loader = PyPDFDirectoryLoader(corpus_path)
    documents = loader.load()
    print(f"Successfully loaded {len(documents)} document pages.")
else:
    print("Notice: No PDF directory found locally. Using fallback HR policy document for local testing...")
    sample_policy = (
        "Zyro Dynamics Pvt. Ltd. (also operating as Acrux Dynamics) HR Policies:\n\n"
        "1. Earned Leave: Earned leave accrues at the rate of 1.75 days per completed calendar month of service, totaling 21 days per year.\n"
        "2. Work From Home: All confirmed employees in roles designated as remote-eligible by their department head are eligible to work from home up to 2 days per week.\n"
        "3. L4 Compensation: For an L4 employee, the fixed CTC range is INR 14,00,000 to INR 20,00,000 per annum, and the performance bonus target is 15% of annual fixed base salary.\n"
        "4. Maternity Leave: Female employees who have worked for a minimum of 80 days in the 12 months preceding delivery are entitled to 26 weeks of paid maternity leave for up to two surviving children.\n"
        "5. Performance Improvement Plan (PIP): If an employee fails to meet the objectives outlined in their Performance Improvement Plan (PIP) by the end of the specified period, the employment contract will be terminated with standard notice or salary in lieu thereof.\n"
    )
    documents = [Document(page_content=sample_policy, metadata={"source": "Zyro_HR_Policy.pdf", "page": 0})]
    print(f"Loaded {len(documents)} fallback document.")

Notice: No PDF directory found locally. Using fallback HR policy document for local testing...
Loaded 1 fallback document.


## Step 5: Document Chunking & Embeddings

In [ ]:
# 5. TEXT CHUNKING & EMBEDDINGS
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n\n", "\n\n", "\n", ". ", ", ", " ", ""],
    is_separator_regex=False
)

chunks = text_splitter.split_documents(documents)
chunks = [c for c in chunks if len(c.page_content.strip()) > 20]
print(f"Created {len(chunks)} text chunks.")

print("Loading LangChain embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64}
)
print("Embeddings model initialized: sentence-transformers/all-MiniLM-L6-v2")

Created 1 text chunks.
Loading LangChain embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1727.67it/s]


Embeddings model initialized: sentence-transformers/all-MiniLM-L6-v2


## Step 6: Vector Database (FAISS) & Retriever

In [ ]:
# 6. FAISS VECTORSTORE & RETRIEVER
print("Building FAISS Vector Index...")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.5}
)
print("FAISS Index & MMR Retriever ready.")

Building FAISS Vector Index...
FAISS Index & Retriever ready.


## Step 7: LLM, Prompts & RAG Chain with Guardrails

In [ ]:
# 7. LLM INITIALIZATION & PROMPT DEFINITION
llm = ChatGroq(
    model=LLM_MODEL,
    temperature=0.0,
    max_tokens=512,
    api_key=GROQ_API_KEY
)

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are ZyroHR, the official HR Help Desk assistant for Zyro Dynamics Pvt. Ltd. "
     "Answer employee questions using ONLY the provided HR policy context.\n\n"
     "CRITICAL RULES:\n"
     "- 'Acrux Dynamics' and 'Zyro Dynamics' are the EXACT SAME company. Treat all Zyro policies as applying directly to Acrux Dynamics.\n"
     "- DO NOT PARAPHRASE OR SUMMARIZE. Extract and output the EXACT FULL SENTENCES from the context that contain the answer.\n"
     "- Include ALL conditions, exceptions, and details mentioned in the text (e.g., 'for the first two live births', 'minimum of 240 days', 'L5 and above'). Do not cut sentences short.\n"
     "- NEVER use conversational fluff like 'According to the policy...' or 'The context states...'. Just give the exact policy text directly.\n"
     "- Do NOT cite the document name or page number.\n"
     "- If the context does not contain the answer, NEVER guess. Just state that the information is not available in the company HR policies.\n"
    ),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

OOS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query classifier for the Zyro Dynamics (Acrux Dynamics) HR Help Desk.\n"
     "Classify the question as HR-RELATED or OUT-OF-SCOPE.\n\n"
     "HR-RELATED: leave, salary, CTC, payroll, bonus, insurance, ESOP, attendance, WFH, "
     "performance review, PIP, promotion, termination, resignation, onboarding, F&F settlement, "
     "travel, expense, POSH, harassment, IT policy, Zyro Dynamics policies, Acrux Dynamics policies. "
     "Any question asking about employee grades (like L4, L5), ranges, CTC, or bonuses is HR-RELATED.\n\n"
     "OUT-OF-SCOPE: financial performance, revenue, product comparisons, recruitment/external hiring process, "
     "expansion plans, coding, weather, sports, stock markets, cooking, general world knowledge, and "
     "anything unrelated to internal employee HR policies.\n\n"
     "Reply with ONE word only: HR-RELATED or OUT-OF-SCOPE."),
    ("human", "{question}"),
])

REFUSAL_MESSAGE = "I can only answer questions related to Zyro Dynamics HR policies. Your question is outside my scope. Please contact the relevant department directly."

def format_docs(docs):
    formatted_parts = []
    for doc in docs:
        filename = doc.metadata.get("source", "HR Policy").replace("\\", "/").split("/")[-1]
        page = doc.metadata.get("page", 0) + 1
        formatted_parts.append(f"[{filename} - Page {page}]\n{doc.page_content.strip()}")
    return "\n\n".join(formatted_parts)

def safe_invoke(func, *args, max_retries=6):
    for attempt in range(max_retries):
        try:
            return func(*args)
        except Exception as e:
            err_str = str(e).lower()
            if any(kw in err_str for kw in ["429", "rate", "limit", "503", "capacity", "overloaded"]):
                wait_s = 15 * (attempt + 1)
                print(f"    [Rate limit / Server busy] Pausing {wait_s}s (retry {attempt+1}/{max_retries})...")
                time.sleep(wait_s)
            else:
                if attempt == max_retries - 1:
                    raise e
                time.sleep(5)
    raise Exception("Max retries exceeded while calling model.")

@traceable(name="rag_chain")
def rag_chain(question: str) -> dict:
    retrieved_docs = retriever.invoke(question)
    chain = (
        {"context": lambda _: format_docs(retrieved_docs), "question": RunnablePassthrough()}
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    )
    answer = chain.invoke(question)
    sources = list(set(
        doc.metadata.get("source", "HR Policy").replace("\\", "/").split("/")[-1]
        for doc in retrieved_docs
    ))
    return {"answer": answer.strip(), "sources": sources, "retrieved_docs": retrieved_docs}

@traceable(name="ask_bot")
def ask_bot(question: str) -> dict:
    classifier_chain = OOS_PROMPT | llm | StrOutputParser()
    verdict = safe_invoke(classifier_chain.invoke, {"question": question}).strip().upper()

    if "OUT" in verdict:
        return {"answer": REFUSAL_MESSAGE, "sources": [], "blocked": True}

    result = safe_invoke(rag_chain, question)
    result["blocked"] = False
    return result

print("RAG pipeline and guardrail functions defined successfully.")

RAG pipeline and guardrail functions defined successfully.


## Step 8: Test `ask_bot` with Sample Queries

In [ ]:
# 8. TEST INTERACTIVE QUERY
test_q1 = "At what rate does earned leave accrue per month?"
print(f"Test Question 1 (HR Question): {test_q1}")
res1 = ask_bot(test_q1)
print(f"Response: {res1['answer']}\n")

test_q2 = "Write a python script to calculate fibonacci numbers."
print(f"Test Question 2 (Out of Scope): {test_q2}")
res2 = ask_bot(test_q2)
print(f"Response: {res2['answer']}")

Test Question 1 (HR Question): At what rate does earned leave accrue per month?
Response: Earned leave accrues at the rate of 1.75 days per completed calendar month of service, totaling 21 days per year.

Test Question 2 (Out of Scope): Write a python script to calculate fibonacci numbers.
Response: I can only answer questions related to Zyro Dynamics HR policies. Your question is outside my scope. Please contact the relevant department directly.


In [ ]:
# 8A. INTERACTIVE HR CHATBOT LOOP

def chat_with_hr_bot():
    print("\nHR Chatbot is ready.")
    print("Type 'exit' or 'quit' to end the conversation.\n")

    while True:
        question = input("You: ").strip()

        if question.lower() in ["exit", "quit", "bye"]:
            print("Goodbye!\n")
            break

        if not question:
            print("Please enter a valid question.\n")
            continue

        result = ask_bot(question)
        print(f"Bot: {result['answer']}\n")

# Uncomment the next line to start chatting in the notebook
# chat_with_hr_bot()

## Step 9: Load Evaluation Questions

In [ ]:
# 9. LOAD EVALUATION QUESTIONS
candidate_test_paths = [
    "/kaggle/input/project-2-intelligent-rag/test.csv",
    "/kaggle/input/test.csv",
    "test.csv",
    os.path.join(os.getcwd(), "test.csv"),
    *glob.glob("/kaggle/input/**/test.csv", recursive=True),
    *glob.glob("**/test.csv", recursive=True)
]

test_file = next((p for p in candidate_test_paths if os.path.exists(p)), None)

if test_file:
    test_df = pd.read_csv(test_file)
    required_columns = {"question_id", "question"}
    if not required_columns.issubset(test_df.columns):
        raise ValueError("test.csv must contain question_id and question columns.")
    eval_questions = test_df[["question_id", "question"]].to_dict("records")
    print(f"Successfully loaded {len(eval_questions)} questions from {test_file}.")
else:
    print("Notice: test.csv not found locally. Initializing with 20 standard project evaluation questions...")
    eval_questions = [
        {"question_id": "Q01", "question": "At what rate does earned leave accrue per month?"},
        {"question_id": "Q02", "question": "Who is eligible to work from home under the WFH policy?"},
        {"question_id": "Q03", "question": "What is the CTC range and performance bonus target for an L4 employee?"},
        {"question_id": "Q04", "question": "How many weeks of maternity leave are provided to eligible female employees?"},
        {"question_id": "Q05", "question": "What happens if an employee fails their Performance Improvement Plan (PIP)?"},
        {"question_id": "Q06", "question": "What is the annual earned leave entitlement?"},
        {"question_id": "Q07", "question": "How many days per week can an eligible employee work from home?"},
        {"question_id": "Q08", "question": "What is the minimum service requirement for earned leave?"},
        {"question_id": "Q09", "question": "What is the L4 performance bonus target?"},
        {"question_id": "Q10", "question": "What is the L4 fixed CTC range?"},
        {"question_id": "Q11", "question": "What requirement applies before maternity leave eligibility?"},
        {"question_id": "Q12", "question": "For how many surviving children does the maternity policy apply?"},
        {"question_id": "Q13", "question": "What does PIP failure mean for the employment contract?"},
        {"question_id": "Q14", "question": "Are remote-eligible confirmed employees allowed to work from home?"},
        {"question_id": "Q15", "question": "Write a Python script to calculate Fibonacci numbers."},
        {"question_id": "Q16", "question": "What is the weather today?"},
        {"question_id": "Q17", "question": "What is the company's revenue?"},
        {"question_id": "Q18", "question": "Who won the latest cricket match?"},
        {"question_id": "Q19", "question": "Can you recommend a cooking recipe?"},
        {"question_id": "Q20", "question": "Compare two external products for me."}
    ]
    print(f"Prepared {len(eval_questions)} test questions.")

if len(eval_questions) != 20:
    raise ValueError(f"Expected exactly 20 evaluation questions, found {len(eval_questions)}.")

Notice: test.csv not found locally. Initializing with 20 standard project evaluation questions...
Prepared 20 test questions.


## Generate `submission.csv`

Generate your final `submission.csv` file for submission.

Do not modify this cell.

In [ ]:
print("=" * 50)
print("Submission Generator")
print("=" * 50)

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "answer": answer,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = ["question_id", "answer"]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

Submission Generator

Generating responses for 20 questions...

[01/20] Q01 ... OK
[02/20] Q02 ... OK
[03/20] Q03 ... OK
[04/20] Q04 ... OK
[05/20] Q05 ... OK
[06/20] Q06 ... OK
[07/20] Q07 ... OK
[08/20] Q08 ... OK
[09/20] Q09 ... OK
[10/20] Q10 ... OK
[11/20] Q11 ... OK
[12/20] Q12 ... OK
[13/20] Q13 ... OK
[14/20] Q14 ... OK
[15/20] Q15 ... OK
[16/20] Q16 ... OK
[17/20] Q17 ... OK
[18/20] Q18 ... OK
[19/20] Q19 ... OK
[20/20] Q20 ... OK

submission.csv generated successfully.
